# NeMo Fine-tuning Pipeline Example

This notebook demonstrates how to use the NeMo fine-tuning pipeline for CodeLlama and Llama3 models.

## Prerequisites

1. NVIDIA GPU with sufficient memory
2. NeMo Framework container or local installation
3. Hugging Face account and token
4. Training data in YAML format

## 1. Setup and Imports

In [ ]:
import os
import sys
import yaml
import json
from pathlib import Path

# Add the current directory to Python path
sys.path.append('.')

from finetune_pipeline import NeMoFineTuningPipeline
from evaluate_model import ModelEvaluator
from configs.model_configs import list_available_models, get_hardware_requirements
from data_preprocessing import convert_yaml_to_jsonl

print("Setup complete!")

## 2. Check Available Models and Hardware Requirements

In [ ]:
# List available models
available_models = list_available_models()
print("Available models:")
for model in available_models:
    print(f"  - {model}")

# Check hardware requirements for a specific model
model_name = "codellama-13b"  # Change this to your desired model
requirements = get_hardware_requirements(model_name)
print(f"\nHardware requirements for {model_name}:")
for key, value in requirements.items():
    print(f"  {key}: {value}")

## 3. Create Sample Training Data

Let's create a sample YAML file with training conversations:

In [ ]:
# Create sample training data
sample_data = [
    {
        "messages": [
            {
                "role": "system",
                "content": "You are an expert test automation assistant using SWTBot."
            },
            {
                "role": "user",
                "content": "Task: Create and build an S32K144 project in Eclipse IDE"
            },
            {
                "role": "assistant",
                "content": "```java\nbot.menu(\"File\").click();\nbot.menu(\"New\").click();\nbot.menu(\"Project...\").click();\nbot.tree().expandNode(\"C/C++\").select(\"C++ Project\");\nbot.button(\"Next >\").click();\n```"
            }
        ]
    },
    {
        "messages": [
            {
                "role": "user",
                "content": "How do I click on a button with SWTBot?"
            },
            {
                "role": "assistant",
                "content": "```java\n// Click button by text\nbot.button(\"OK\").click();\n\n// Click button by tooltip\nbot.buttonWithTooltip(\"Save file\").click();\n\n// Click button by ID\nbot.buttonWithId(\"saveButton\").click();\n```"
            }
        ]
    },
    {
        "messages": [
            {
                "role": "user",
                "content": "Show me how to work with text fields in SWTBot"
            },
            {
                "role": "assistant",
                "content": "```java\n// Type text in a text field\nbot.textWithLabel(\"Project name:\").setText(\"MyProject\");\n\n// Get text from a text field\nString projectName = bot.textWithLabel(\"Project name:\").getText();\n\n// Clear text field\nbot.textWithLabel(\"Project name:\").setText(\"\");\n```"
            }
        ]
    }
]

# Save sample data to YAML file
sample_data_path = "sample_training_data.yaml"
with open(sample_data_path, 'w') as f:
    yaml.dump_all(sample_data, f, default_flow_style=False)

print(f"Sample training data saved to {sample_data_path}")
print(f"Number of conversations: {len(sample_data)}")

## 4. Data Preprocessing

Convert the YAML data to JSONL format required by NeMo:

In [ ]:
# Convert YAML to JSONL
output_base = "processed_training_data"
train_file, val_file = convert_yaml_to_jsonl(
    sample_data_path, 
    output_base, 
    validation_split=0.2  # 20% for validation
)

print(f"Training file: {train_file}")
print(f"Validation file: {val_file}")

# Show a sample from the training data
with open(train_file, 'r') as f:
    sample_line = f.readline()
    sample_data = json.loads(sample_line)
    print("\nSample training data:")
    print(json.dumps(sample_data, indent=2))

## 5. Initialize Fine-tuning Pipeline

**Note**: The following cells demonstrate the pipeline setup. For actual training, you'll need:
- Sufficient GPU memory
- Hugging Face token
- More training data for meaningful results

In [ ]:
# Initialize the pipeline
model_name = "codellama-13b"  # Change to your desired model
output_dir = "./pipeline_outputs"

pipeline = NeMoFineTuningPipeline(model_name, output_dir)
print(f"Pipeline initialized for {model_name}")
print(f"Output directory: {output_dir}")

## 6. Configuration Preview

Let's see what the training configuration looks like:

In [ ]:
# Create a sample config to see the structure
# Note: This won't work without actual model files, but shows the config structure
try:
    config_file = pipeline.create_config_file(
        train_file, 
        val_file, 
        "dummy_model_path.nemo"  # Placeholder path
    )
    
    # Display the configuration
    with open(config_file, 'r') as f:
        config = yaml.safe_load(f)
    
    print("Training configuration preview:")
    print(f"Model: {config['name']}")
    print(f"Devices: {config['trainer']['devices']}")
    print(f"Max steps: {config['trainer']['max_steps']}")
    print(f"Learning rate: {config['model']['optim']['lr']}")
    print(f"LoRA adapter dim: {config['model']['peft']['lora_tuning']['adapter_dim']}")
    
except Exception as e:
    print(f"Config preview failed (expected without actual model): {e}")

## 7. Running the Complete Pipeline

**Important**: This cell shows how to run the complete pipeline. 
Uncomment and modify the code below when you're ready to run actual training.

**Requirements**:
- Set your Hugging Face token
- Ensure sufficient GPU memory
- Have substantial training data

In [ ]:
# UNCOMMENT AND MODIFY THE CODE BELOW FOR ACTUAL TRAINING

# # Set your Hugging Face token
# HF_TOKEN = "your_huggingface_token_here"
# 
# # Run the complete pipeline
# try:
#     final_model_path = pipeline.run_full_pipeline(
#         yaml_data_path=sample_data_path,
#         max_steps=50,  # Use more steps for real training
#         hf_token=HF_TOKEN,
#         validation_split=0.1
#     )
#     
#     print(f"Training completed successfully!")
#     print(f"Final model saved to: {final_model_path}")
#     
# except Exception as e:
#     print(f"Training failed: {e}")

print("Pipeline code ready - uncomment and set HF_TOKEN to run actual training")

## 8. Model Evaluation

After training, you can evaluate your model:

In [ ]:
# UNCOMMENT THE CODE BELOW AFTER TRAINING IS COMPLETE

# # Initialize evaluator
# model_path = "path/to/your/trained/model.nemo"
# evaluator = ModelEvaluator(model_path, model_name, "./evaluation_results")
# 
# # Run comprehensive evaluation
# results = evaluator.run_comprehensive_evaluation(
#     test_data_path=val_file,
#     tokens_to_generate=100
# )
# 
# print("Evaluation Results:")
# print(json.dumps(results, indent=2))

print("Evaluation code ready - uncomment after training is complete")

## 9. Command Line Usage Examples

You can also use the pipeline from the command line:

In [ ]:
print("Command line usage examples:")
print()
print("1. Check hardware requirements:")
print("   python finetune_pipeline.py --model codellama-13b --check-hardware")
print()
print("2. Run complete fine-tuning pipeline:")
print("   python finetune_pipeline.py \\")
print("       --model codellama-13b \\")
print("       --data sample_training_data.yaml \\")
print("       --output-dir ./outputs \\")
print("       --max-steps 100 \\")
print("       --hf-token your_token_here")
print()
print("3. Evaluate trained model:")
print("   python evaluate_model.py \\")
print("       --model-path ./outputs/models/codellama-13b_merged.nemo \\")
print("       --model-name codellama-13b \\")
print("       --test-data processed_training_data.val.jsonl \\")
print("       --eval-type comprehensive")
print()
print("4. Data preprocessing only:")
print("   python data_preprocessing.py \\")
print("       --input sample_training_data.yaml \\")
print("       --output processed_data \\")
print("       --validation-split 0.1")

## 10. Tips and Best Practices

### Data Preparation
- Use high-quality, diverse training conversations
- Ensure consistent formatting in your YAML files
- Include system prompts to guide model behavior
- Balance conversation lengths and complexity

### Training
- Start with smaller models for experimentation
- Monitor GPU memory usage and adjust batch sizes
- Use validation data to prevent overfitting
- Save checkpoints regularly for long training runs

### Evaluation
- Test on diverse, unseen data
- Use multiple evaluation metrics
- Compare with baseline models
- Perform human evaluation for code quality

### Hardware Optimization
- Use mixed precision training (bf16)
- Enable gradient checkpointing for memory efficiency
- Optimize tensor and pipeline parallelism
- Consider using Flash Attention for large models

## Next Steps

1. **Prepare your actual training data** in YAML format
2. **Set up your Hugging Face token** for model access
3. **Check hardware requirements** for your chosen model
4. **Run the pipeline** with your data
5. **Evaluate the results** and iterate on your approach

For more information, see the README.md file and configuration examples in the `configs/` directory.